# การตรวจจับไฟจากภาพถ่ายทางอากาศด้วย YOLO26: การฝึกโมเดลและการนำไปใช้งาน

<a href="https://colab.research.google.com/github/jakkzz/Fire-Detection-Drone/blob/main/th/drone_fire_detection_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🇹🇭 **ภาษาไทย** (เอกสารฉบับนี้) · [🇬🇧 English](https://github.com/jakkzz/Fire-Detection-Drone/blob/main/drone_fire_detection_yolo26.ipynb)

โน้ตบุ๊กฉบับนี้นำเสนอกระบวนการฝึกโมเดลตรวจจับไฟจากภาพถ่ายทางอากาศที่บันทึกด้วยอากาศยานไร้คนขับ
โดยใช้ [Ultralytics YOLO26](https://docs.ultralytics.com/models/yolo26/)
พร้อมทั้งการนำโมเดลที่ได้ไปประยุกต์ใช้กับทั้งภาพนิ่งและวิดีโอ

เนื้อหาแบ่งออกเป็นสามส่วนภายในไฟล์เดียว **ควรดำเนินการตามลำดับจากบนลงล่าง**
เนื่องจากส่วนที่ 2 และส่วนที่ 3 จะเรียกใช้ไฟล์ `best.pt` ที่ได้จากส่วนที่ 1 โดยอัตโนมัติ
ผู้ใช้จึงไม่จำเป็นต้องอัปโหลดไฟล์ดังกล่าวด้วยตนเอง

| ส่วน | เนื้อหา | GPU |
|---|---|---|
| [ส่วนที่ 1](#part-1) | ฝึกโมเดล YOLO26 ด้วยชุดข้อมูลจาก Roboflow | จำเป็น |
| [ส่วนที่ 2](#part-2) | ตรวจจับไฟบนภาพนิ่ง และแสดงผลด้วย Supervision | ไม่จำเป็น |
| [ส่วนที่ 3](#part-3) | ตรวจจับและติดตามไฟบนวิดีโอด้วย ByteTrack | จำเป็น |

**การตั้งค่า runtime:** `Runtime` → `Change runtime type` → **T4 GPU** (หรือสูงกว่า) แล้วเลือก `Save`

<a id="part-1"></a>
<hr>

<h1 align="center">🔥 ส่วนที่ 1 · การฝึกโมเดล</h1>

<p align="center">
  <b>Ultralytics YOLO26 + Roboflow</b> — ปรับละเอียด (fine-tune) โมเดลตรวจจับไฟด้วยชุดข้อมูลภาพถ่ายทางอากาศ<br>
  <i>ส่วนนี้จำเป็นต้องใช้ GPU และเป็นขั้นตอนที่ใช้เวลามากที่สุด</i>
</p>

<hr>

## 1. ตรวจสอบ GPU

In [ ]:
!nvidia-smi

## 2. ติดตั้งไลบรารี

การกำหนดเวอร์ชันในที่นี้ระบุเป็นเวอร์ชันขั้นต่ำ มิได้ตรึงไว้ที่เวอร์ชันใดเวอร์ชันหนึ่ง
เพื่อให้แน่ใจว่า API ที่โน้ตบุ๊กฉบับนี้เรียกใช้มีอยู่จริง ขณะเดียวกันก็เปิดให้ pip
เลือกเวอร์ชันที่เข้ากันได้กับ PyTorch รุ่นที่ Colab ติดตั้งมาให้

การติดตั้งดำเนินการครั้งเดียวให้ครบทั้งสามส่วน เพื่อไม่ต้องติดตั้งเพิ่มเติมระหว่างการทำงาน
โดยเฉพาะ `lap` ซึ่งตัวติดตามวัตถุ ByteTrack ในส่วนที่ 3 จำเป็นต้องใช้ แต่ Ultralytics
ไม่ได้กำหนดไว้เป็นแพ็กเกจที่ต้องพึ่งพา (dependency) หากไม่ติดตั้งไว้ล่วงหน้า
`model.track(...)` จะเรียก pip ติดตั้งเองในระหว่างการประมวลผลวิดีโอ

In [ ]:
%pip install -q "ultralytics>=8.4.122" "supervision>=0.30.0" "lap>=0.5.12" "roboflow>=1.4.1"

import supervision as sv
import ultralytics

ultralytics.checks()
print("supervision:", sv.__version__)

In [ ]:
import os
from pathlib import Path

from ultralytics import YOLO

HOME = Path.cwd()
print("HOME:", HOME)

## 3. ดาวน์โหลดชุดข้อมูลจาก Roboflow

ขั้นตอนนี้จำเป็นต้องใช้ Roboflow API key ซึ่งขอได้โดยไม่มีค่าใช้จ่ายที่
<https://app.roboflow.com/settings/api>

หากทำงานบน Colab ควรบันทึก key ไว้ในแผง 🔑 **Secrets** ทางแถบด้านซ้าย ภายใต้ชื่อ
`ROBOFLOW_API_KEY` และเปิดสิทธิ์ *Notebook access* เซลล์ด้านล่างจะอ่านค่าจากแหล่งดังกล่าวก่อน
หากไม่พบจะอ่านจากตัวแปรสภาพแวดล้อม และสอบถามจากผู้ใช้เป็นลำดับสุดท้าย
แนวทางนี้ช่วยป้องกันไม่ให้ key ถูกบันทึกลงในไฟล์โน้ตบุ๊กเมื่อ commit

In [ ]:
import getpass
import os


def get_roboflow_api_key() -> str:
    """อ่าน API key จาก Colab Secrets เป็นลำดับแรก หากไม่พบจะอ่านจากตัวแปรสภาพแวดล้อม และสอบถามจากผู้ใช้เป็นลำดับสุดท้าย"""
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("ROBOFLOW_API_KEY")
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get("ROBOFLOW_API_KEY")
    if key:
        return key
    return getpass.getpass("Roboflow API key: ")


ROBOFLOW_API_KEY = get_roboflow_api_key()

In [ ]:
from roboflow import Roboflow

DATASETS_DIR = HOME / "datasets"
DATASETS_DIR.mkdir(exist_ok=True)
os.chdir(DATASETS_DIR)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("tim-4ijf0").project("drone-fire-detection-byija")

# "yolov8" เป็นชื่อ *รูปแบบการจัดวางไฟล์สำหรับการส่งออก* ชุดข้อมูล (ภาพ ไฟล์ label .txt
# ตามมาตรฐาน YOLO และ data.yaml) มิใช่เวอร์ชันของโมเดล ทั้งนี้ Roboflow ไม่มีรูปแบบการส่งออก
# ชื่อ "yolo26" และไม่จำเป็นต้องมี เนื่องจากรูปแบบดังกล่าวคือรูปแบบมาตรฐานของ YOLO ซึ่ง YOLO26
# นำไปใช้ฝึกได้โดยตรง สำหรับ data.yaml ที่ส่งออกมาจะระบุพาธแบบสัมพัทธ์ "../train/images"
# ซึ่ง Ultralytics จะค้นหาโดยอ้างอิงจากโฟลเดอร์ของ data.yaml เอง จึงไม่จำเป็นต้องปรับแก้พาธเพิ่มเติม
dataset = project.version(1).download("yolov8")

os.chdir(HOME)

DATA_YAML = Path(dataset.location) / "data.yaml"
print("data.yaml:", DATA_YAML)
print(DATA_YAML.read_text())

## 4. ฝึกโมเดล

YOLO26 เป็นสถาปัตยกรรมแบบครบวงจร (end-to-end) ที่ไม่ผ่านขั้นตอน NMS จึงไม่มีค่าขีดแบ่ง `iou`
ของ NMS ให้ต้องปรับ เนื่องจากโมเดลให้ผลลัพธ์เป็นกรอบสุดท้ายโดยตรง

โค้ดส่วนนี้เก็บค่า `results.save_dir` ไว้ เพื่อให้เซลล์ถัดไปไม่ต้องระบุพาธ `runs/detect/train`
แบบตายตัว เนื่องจาก Ultralytics จะเพิ่มลำดับเป็น `train2`, `train3`, … ในการเรียกใช้ครั้งต่อ ๆ ไป

หากพบปัญหาหน่วยความจำไม่เพียงพอ (out of memory) ให้ลดค่า `imgsz` เป็น 640
หรือเปลี่ยน `model` เป็น `yolo26n.pt`

In [ ]:
MODEL_ARCH = "yolo26m.pt"  # n / s / m / l / x

model = YOLO(MODEL_ARCH)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=800,
    plots=True,
)

RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
print("run dir:", RUN_DIR)
print("best weights:", BEST_WEIGHTS)

In [ ]:
from IPython.display import Image, display

for artifact in ["confusion_matrix.png", "results.png", "val_batch0_pred.jpg"]:
    path = RUN_DIR / artifact
    if path.exists():
        print(artifact)
        display(Image(filename=str(path), width=700))
    else:
        print("ไม่พบไฟล์:", artifact)

## 5. ตรวจสอบความแม่นยำของโมเดล (validate)

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))
metrics = best_model.val(data=str(DATA_YAML))

print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP75:    {metrics.box.map75:.4f}")

## 6. ทำนายผลบนชุดทดสอบ (test split)

In [ ]:
predict_results = best_model.predict(
    source=str(Path(dataset.location) / "test" / "images"),
    conf=0.25,
    save=True,
)

PREDICT_DIR = Path(predict_results[0].save_dir)
print("predictions:", PREDICT_DIR)

In [ ]:
import glob

for image_path in sorted(glob.glob(f"{PREDICT_DIR}/*.jpg"))[:3]:
    display(Image(filename=image_path, width=700))

## 7. ส่งออกไฟล์ weights

ส่วนที่ 2 และส่วนที่ 3 ของโน้ตบุ๊กฉบับนี้จะเรียกใช้ `best.pt` ผ่านตัวแปร `BEST_WEIGHTS` โดยอัตโนมัติ
เซลล์นี้จึงเป็นทางเลือกเสริม สำหรับกรณีที่ต้องการดาวน์โหลด checkpoint ไปใช้งานภายนอก

In [ ]:
try:
    from google.colab import files  # type: ignore

    files.download(str(BEST_WEIGHTS))
except ImportError:
    print("ไม่ได้ทำงานบน Colab ไฟล์ weights อยู่ที่:", BEST_WEIGHTS)

<a id="part-2"></a>
<hr>

<h1 align="center">🖼️ ส่วนที่ 2 · การตรวจจับไฟบนภาพนิ่ง</h1>

<p align="center">
  <b>YOLO26 + Supervision</b> — ประมวลผลภาพนิ่งหนึ่งภาพ พร้อมแสดงกรอบและป้ายกำกับผลการตรวจจับ<br>
  <i>ส่วนนี้ไม่จำเป็นต้องใช้ GPU เนื่องจากการประมวลผลภาพเดียวใช้เวลาไม่นานบน CPU</i>
</p>

<hr>

## 8. เตรียมไฟล์ weights และภาพทดสอบ

หากดำเนินการส่วนที่ 1 มาแล้วในเซสชันเดียวกัน เซลล์นี้จะเรียกใช้ `best.pt` ที่ได้จากการฝึกทันที
แต่หากเปิดโน้ตบุ๊กขึ้นมาใหม่แล้วเริ่มที่ส่วนนี้ ระบบจะค้นหา `best.pt` ในโฟลเดอร์ปัจจุบันก่อน
และขอให้ผู้ใช้อัปโหลดเป็นลำดับสุดท้าย สำหรับภาพตัวอย่างนั้นจะดาวน์โหลดจากคลังโค้ดโดยอัตโนมัติ

In [ ]:
from pathlib import Path


def resolve_best_weights() -> Path:
    """ค้นหา best.pt ตามลำดับ: ไฟล์ที่ได้จากการฝึกในเซสชันนี้ → ไฟล์ในโฟลเดอร์ปัจจุบัน → ขอให้ผู้ใช้อัปโหลด"""
    trained = globals().get("BEST_WEIGHTS")
    if trained is not None and Path(trained).exists():
        return Path(trained)

    local = Path("best.pt")
    if local.exists():
        return local

    print("ไม่พบ best.pt โปรดอัปโหลดไฟล์ หรือย้อนกลับไปดำเนินการส่วนที่ 1")
    try:
        from google.colab import files  # type: ignore

        files.upload()
    except ImportError:
        raise FileNotFoundError("โปรดวางไฟล์ best.pt ไว้ในโฟลเดอร์เดียวกับโน้ตบุ๊กฉบับนี้")
    if not local.exists():
        raise FileNotFoundError("ไม่พบไฟล์ best.pt หลังการอัปโหลด")
    return local


MODEL_PATH = resolve_best_weights()

IMAGE_PATH = Path("fire_image.png")
if not IMAGE_PATH.exists():
    !wget -q -O {IMAGE_PATH} https://raw.githubusercontent.com/jakkzz/Fire-Detection-Drone/main/fire_image.png

print("model:", MODEL_PATH.resolve())
print("image:", IMAGE_PATH.resolve())

In [ ]:
model

In [ ]:
image

In [ ]:
import cv2
import supervision as sv

# Display the image as is. If it now shows correct colors, it implies 'image' was already RGB.
sv.plot_image(image=image, size=(10, 10))

## 9. ประมวลผลภาพนิ่ง

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))

image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise FileNotFoundError(f"ไม่สามารถอ่านไฟล์ {IMAGE_PATH} ได้")

# YOLO รับภาพในรูปแบบ BGR จึงใช้ภาพตามที่ OpenCV อ่านมาได้โดยตรง

result = model(image, conf=0.25, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)

print("detections:", len(detections))

## 10. แสดงกรอบและป้ายกำกับ

Supervision รุ่นปัจจุบันแยกการแสดงผลออกเป็น annotator หลายตัว โดย `BoxAnnotator`
ไม่รับอาร์กิวเมนต์ `labels=` อีกต่อไป (ถูกถอดออกตั้งแต่ supervision 0.22)
การแสดงป้ายกำกับจึงเป็นหน้าที่ของ `LabelAnnotator`

ชื่อชั้นข้อมูลอ่านจาก `detections["class_name"]` ซึ่ง `Detections.from_ultralytics`
กำหนดให้จากตัวโมเดลโดยตรง จึงไม่จำเป็นต้องกำหนดรายชื่อชั้นข้อมูลเอง
ซึ่งมีความเสี่ยงที่จะไม่สอดคล้องกับ weights ที่ใช้งานอยู่

In [ ]:
box_annotator = sv.BoxAnnotator(thickness=3)
label_annotator = sv.LabelAnnotator(text_scale=0.8, text_thickness=2, text_padding=6)

labels = [
    f"{class_name} {confidence:.2f}"
    for class_name, confidence
    in zip(detections["class_name"], detections.confidence)
]

# วาดผลลัพธ์ลงบนภาพ BGR ตามที่ OpenCV อ่านมา
annotated = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

In [ ]:
sv.plot_image(image=annotated, size=(10, 10))

## 11. บันทึกภาพผลลัพธ์

In [ ]:
OUTPUT_PATH = Path("fire_image_annotated.png")
cv2.imwrite(str(OUTPUT_PATH), annotated)
print("saved:", OUTPUT_PATH.resolve())

<a id="part-3"></a>
<hr>

<h1 align="center">🎥 ส่วนที่ 3 · การตรวจจับและติดตามไฟบนวิดีโอ</h1>

<p align="center">
  <b>YOLO26 + ByteTrack + Supervision</b> — ประมวลผลตลอดทั้งวิดีโอ ติดตามไฟแต่ละจุดข้ามเฟรม และบันทึกเป็นวิดีโอผลลัพธ์<br>
  <i>ส่วนนี้ควรใช้ GPU เนื่องจากการประมวลผลวิดีโอบน CPU ใช้เวลานาน</i>
</p>

<hr>

> **ข้อสังเกตเกี่ยวกับการเปลี่ยนแปลง** เดิมการติดตามวัตถุจำเป็นต้องโคลนคลังโค้ด
> [ByteTrack](https://github.com/ifzhang/ByteTrack) มาคอมไพล์ YOLOX จากซอร์ส และติดตั้ง
> `onemetric` กับ `cython_bbox` ซึ่งชุดเครื่องมือดังกล่าวไม่สามารถคอมไพล์บน Python
> รุ่นปัจจุบันได้อีกต่อไป ปัจจุบัน ByteTrack รวมอยู่ใน Ultralytics แล้ว การเรียก
> `model.track(...)` จึงดำเนินการให้ครบถ้วนโดยไม่ต้องติดตั้งเพิ่มเติม และ Supervision
> สามารถอ่านหมายเลข track จากผลลัพธ์ได้โดยตรง

## 12. เตรียมวิดีโอต้นทาง

ส่วนนี้ใช้ไฟล์ `best.pt` ชุดเดียวกับส่วนที่ 2 สำหรับวิดีโอตัวอย่าง `fire.mp4`
จะดาวน์โหลดจากคลังโค้ดโดยอัตโนมัติ

In [ ]:
SOURCE_VIDEO_PATH = Path("fire.mp4")
TARGET_VIDEO_PATH = Path("fire_result.mp4")

if not SOURCE_VIDEO_PATH.exists():
    !wget -q -O {SOURCE_VIDEO_PATH} https://github.com/jakkzz/Fire-Detection-Drone/raw/main/fire.mp4

MODEL_PATH = resolve_best_weights()

video_info = sv.VideoInfo.from_video_path(str(SOURCE_VIDEO_PATH))
print("model:", MODEL_PATH.resolve())
print(video_info)

### 12.1. ดูตัวอย่างเฟรมจากวิดีโอต้นทาง

สุ่มเก็บเฟรมจากวิดีโอมาแสดงเป็นตาราง เพื่อสำรวจเนื้อหาตลอดคลิปโดยไม่ต้องฝังไฟล์วิดีโอทั้งไฟล์ไว้ในโน้ตบุ๊ก

`sv.plot_images_grid` และ `sv.plot_image` แปลง BGR เป็น RGB ให้เองอยู่แล้ว
จึงส่งเฟรมที่ได้จาก `get_video_frames_generator` เข้าไปได้โดยตรง โดยไม่ต้องเรียก `cv2.cvtColor` ก่อน
มิเช่นนั้นภาพจะถูกแปลงซ้ำสองครั้ง และสีจะสลับกัน

In [ ]:
stride = video_info.total_frames // 6
frames = list(sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH), stride=stride))[:6]

sv.plot_images_grid(
    images=frames,  # ภาพ BGR จาก OpenCV ส่งเข้าไปได้โดยตรง
    grid_size=(2, 3),
    titles=[f"t={i * stride / video_info.fps:.1f}s" for i in range(len(frames))],
    size=(12, 6),
)

## 13. โหลดโมเดลสำหรับการประมวลผลวิดีโอ

In [ ]:
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))
model.fuse()

print("classes:", model.names)

## 14. ทดสอบการแสดงผลบนเฟรมเดียว

ขั้นตอนนี้เป็นการตรวจสอบความถูกต้องด้วยต้นทุนต่ำ ก่อนจะประมวลผลตลอดทั้งวิดีโอซึ่งใช้เวลานาน

In [ ]:
import cv2

CONFIDENCE_THRESHOLD = 0.25

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.6, text_thickness=2, text_padding=5)


def make_labels(detections: sv.Detections) -> list[str]:
    """สร้างป้ายกำกับในรูปแบบ `#id class conf` โดยละหมายเลข id เมื่อยังไม่ได้เปิดใช้การติดตามวัตถุ"""
    labels = []
    for i in range(len(detections)):
        name = detections["class_name"][i]
        conf = detections.confidence[i]
        tracker_id = None if detections.tracker_id is None else detections.tracker_id[i]
        prefix = "" if tracker_id is None else f"#{tracker_id} "
        labels.append(f"{prefix}{name} {conf:.2f}")
    return labels


frame = next(sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH)))

result = model(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
detections = sv.Detections.from_ultralytics(result)

annotated = box_annotator.annotate(scene=frame.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=make_labels(detections))

# sv.plot_image แปลง BGR เป็น RGB ให้เองอยู่แล้ว จึงส่งภาพ BGR เข้าไปได้โดยตรง
sv.plot_image(image=annotated, size=(10, 10))

## 15. การตรวจจับและติดตามวัตถุตลอดทั้งวิดีโอ

การเรียก `model.track(..., persist=True, tracker="bytetrack.yaml")` จะรักษาสถานะของตัวติดตาม
ไว้ระหว่างการเรียกแต่ละครั้ง ไฟแต่ละจุดจึงได้รับหมายเลข id ที่คงที่ข้ามเฟรม จากนั้น
`sv.Detections.from_ultralytics` จะอ่านหมายเลขดังกล่าวมาเก็บไว้ใน `detections.tracker_id`
และ `TraceAnnotator` จะนำไปแสดงเป็นเส้นแสดงร่องรอยการเคลื่อนที่

ByteTrack ออกแบบมาให้รับผลการตรวจจับที่มีค่าความเชื่อมั่น *ต่ำ* ด้วย เนื่องจากอัลกอริทึม
จะนำกรอบที่มีค่าความเชื่อมั่นต่ำกลับมาจับคู่กับ track ที่มีอยู่เดิม ซึ่งเป็นที่มาของ
ความแม่นยำโดยส่วนใหญ่ โค้ดส่วนนี้จึงกำหนดค่า `conf` ต่ำสำหรับ `track()`
แล้วจึงกรองด้วยค่าความเชื่อมั่นที่ผลลัพธ์ในภายหลัง แทนที่จะตัดข้อมูลที่ตัวติดตามจำเป็นต้องใช้
ออกตั้งแต่ต้นทาง

In [ ]:
from tqdm.auto import tqdm

trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=30)

TRACKER_INPUT_CONF = 0.1  # กำหนดค่าต่ำโดยเจตนา: ByteTrack ใช้กรอบที่มีความเชื่อมั่นต่ำจับคู่กับ track ที่ดำเนินอยู่

frame_generator = sv.get_video_frames_generator(str(SOURCE_VIDEO_PATH))

with sv.VideoSink(str(TARGET_VIDEO_PATH), video_info) as sink:
    for frame in tqdm(frame_generator, total=video_info.total_frames):
        result = model.track(
            frame,
            conf=TRACKER_INPUT_CONF,
            persist=True,
            tracker="bytetrack.yaml",
            verbose=False,
        )[0]
        detections = sv.Detections.from_ultralytics(result)
        # ติดตามโดยใช้กรอบที่มีความเชื่อมั่นต่ำด้วย แต่แสดงผลเฉพาะกรอบที่มีความเชื่อมั่นเพียงพอ
        detections = detections[detections.confidence >= CONFIDENCE_THRESHOLD]

        annotated = frame.copy()
        if detections.tracker_id is not None:
            annotated = trace_annotator.annotate(scene=annotated, detections=detections)
        annotated = box_annotator.annotate(scene=annotated, detections=detections)
        annotated = label_annotator.annotate(
            scene=annotated, detections=detections, labels=make_labels(detections)
        )

        sink.write_frame(annotated)

print("wrote:", TARGET_VIDEO_PATH.resolve())

### 15.1. ฟังก์ชันช่วยสำหรับการประมวลผลวิดีโอ

นี่คือฟังก์ชันที่รวมขั้นตอนการตรวจจับและติดตามวัตถุในวิดีโอ การใส่กรอบและป้ายกำกับ และบันทึกผลลัพธ์เป็นไฟล์วิดีโอใหม่

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import supervision as sv
from tqdm.auto import tqdm

def process_and_save_video(
    model: YOLO,
    source_video_path: Path,
    target_video_path: Path,
    confidence_threshold: float,
    tracker_input_conf: float,
) -> None:
    """
    Processes a video with object detection and tracking, then saves the annotated video.

    Args:
        model (YOLO): The loaded YOLO model for object detection and tracking.
        source_video_path (Path): Path to the input video file.
        target_video_path (Path): Path to save the output annotated video file.
        confidence_threshold (float): Minimum confidence score for detections to be displayed.
        tracker_input_conf (float): Confidence threshold for the ByteTrack input (can be lower).
    """
    video_info = sv.VideoInfo.from_video_path(str(source_video_path))
    frame_generator = sv.get_video_frames_generator(str(source_video_path))

    # Initialize annotators and make_labels function within the function for robustness
    box_annotator = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator(text_scale=0.6, text_thickness=2, text_padding=5)
    trace_annotator = sv.TraceAnnotator(thickness=2, trace_length=30)

    def make_labels(detections: sv.Detections) -> list[str]:
        """สร้างป้ายกำกับในรูปแบบ `#id class conf` โดยละหมายเลข id เมื่อยังไม่ได้เปิดใช้การติดตามวัตถุ"""
        labels = []
        for i in range(len(detections)):
            name = detections["class_name"][i]
            conf = detections.confidence[i]
            tracker_id = None if detections.tracker_id is None else detections.tracker_id[i]
            prefix = "" if tracker_id is None else f"#{tracker_id} "
            labels.append(f"{prefix}{name} {conf:.2f}")
        return labels

    with sv.VideoSink(str(target_video_path), video_info) as sink:
        for frame in tqdm(frame_generator, total=video_info.total_frames):
            result = model.track(
                frame,
                conf=tracker_input_conf,
                persist=True,
                tracker="bytetrack.yaml",
                verbose=False,
            )[0]
            detections = sv.Detections.from_ultralytics(result)
            detections = detections[detections.confidence >= confidence_threshold]

            annotated = frame.copy()
            if detections.tracker_id is not None:
                annotated = trace_annotator.annotate(scene=annotated, detections=detections)
            annotated = box_annotator.annotate(scene=annotated, detections=detections)
            annotated = label_annotator.annotate(
                scene=annotated, detections=detections, labels=make_labels(detections)
            )

            sink.write_frame(annotated)

    print(f"Annotated video saved to: {target_video_path.resolve()}")


# Example usage:
# Note: This example will create a new output video file for demonstration purposes.
DEMO_TARGET_VIDEO_PATH = Path("fire_result_demo.mp4")

# Explicitly define these for the example's robustness, if not already defined globally.
# These values are taken from cells 36 and 38.
example_confidence_threshold = 0.25
example_tracker_input_conf = 0.1

process_and_save_video(
    model=model,
    source_video_path=SOURCE_VIDEO_PATH,
    target_video_path=DEMO_TARGET_VIDEO_PATH,
    confidence_threshold=example_confidence_threshold,
    tracker_input_conf=example_tracker_input_conf,
)

print(f"Demonstration video saved to: {DEMO_TARGET_VIDEO_PATH.resolve()}")


## 16. แสดงวิดีโอผลลัพธ์ในโน้ตบุ๊ก

Supervision บันทึกไฟล์ผลลัพธ์ด้วยตัวเข้ารหัส `mp4v` ซึ่งเบราว์เซอร์ไม่สามารถถอดรหัสได้
จึงต้องแปลงเป็น H.264 ด้วย ffmpeg (Colab ติดตั้งมาให้แล้ว) วิดีโอจึงจะแสดงผลในโน้ตบุ๊กได้

In [ ]:
PLAYABLE_VIDEO_PATH = Path("fire_result_demo.mp4")

!ffmpeg -y -loglevel error -i {TARGET_VIDEO_PATH} -vcodec libx264 -pix_fmt yuv420p {PLAYABLE_VIDEO_PATH}

print("wrote:", PLAYABLE_VIDEO_PATH.resolve())

In [ ]:
import base64

from IPython.display import HTML, display

encoded = base64.b64encode(PLAYABLE_VIDEO_PATH.read_bytes()).decode()
display(HTML(f'''
<video width="720" controls>
  <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
</video>
'''))

## 17. ดาวน์โหลดผลลัพธ์

In [ ]:
try:
    from google.colab import files  # type: ignore

    files.download(str(PLAYABLE_VIDEO_PATH))
except ImportError:
    print("ไม่ได้ทำงานบน Colab ไฟล์ผลลัพธ์อยู่ที่:", PLAYABLE_VIDEO_PATH.resolve())